# Customer Churn Prediction — Data Preprocessing

In this notebook, we will prepare the customer churn dataset for machine learning.

Steps:

1. Load the dataset
2. Inspect the required columns
3. Clean `TotalCharges`
4. Remove unnecessary columns
5. Separate features and target
6. Perform train-test split
7. Identify numerical and categorical features
8. Build the preprocessing pipeline
9. Transform the training and testing data
10. Validate the transformed data

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [3]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.shape

(7043, 21)

In [5]:
df["TotalCharges"].dtype

<StringDtype(storage='python', na_value=nan)>

In [6]:
df["TotalCharges"].isna().sum()

np.int64(0)

In [7]:
(df["TotalCharges"].str.strip() == "").sum()

np.int64(11)

In [8]:
df.loc[df["TotalCharges"].str.strip() == "", ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,,No
753,0,20.25,,No
936,0,80.85,,No
1082,0,25.75,,No
1340,0,56.05,,No
3331,0,19.85,,No
3826,0,25.35,,No
4380,0,20.00,,No
5218,0,19.70,,No
6670,0,73.35,,No


In [9]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [10]:
df["TotalCharges"].dtype

dtype('float64')

In [11]:
df["TotalCharges"].isna().sum()

np.int64(11)

In [12]:
df.drop(columns=["customerID"], inplace=True)

In [14]:
X=df.drop(columns=["Churn"])
y=df["Churn"]

In [15]:
y=y.map({"Yes": 1, "No": 0})

In [16]:
y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [19]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)


In [21]:
print("Training churn distribution:")
print(y_train.value_counts(normalize=True)*100)

Training churn distribution:
Churn
0    73.464679
1    26.535321
Name: proportion, dtype: float64


In [22]:
print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True)*100)


Testing churn distribution:
Churn
0    73.456352
1    26.543648
Name: proportion, dtype: float64


In [25]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_features

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

In [27]:
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()
categorical_features

C:\Users\ANEES\AppData\Local\Temp\ipykernel_31128\722448846.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

In [29]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore",sparse_output=False))
    ]
)

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [31]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_train_transformed.shape

(5634, 45)

In [33]:
X_test_transformed = preprocessor.transform(X_test)
X_test_transformed.shape

(1409, 45)

In [34]:
print("Original X_train shape:", X_train.shape)
print("Transformed X_train shape:", X_train_transformed.shape)

print("Original X_test shape:", X_test.shape)
print("Transformed X_test shape:", X_test_transformed.shape)

Original X_train shape: (5634, 19)
Transformed X_train shape: (5634, 45)
Original X_test shape: (1409, 19)
Transformed X_test shape: (1409, 45)


In [35]:
np.isnan(X_train_transformed).sum()

np.int64(0)

In [36]:
np.isnan(X_test_transformed).sum()

np.int64(0)

In [37]:
feature_names = preprocessor.get_feature_names_out()

In [38]:
len(feature_names)

45

In [39]:
feature_names

array(['num__SeniorCitizen', 'num__tenure', 'num__MonthlyCharges',
       'num__TotalCharges', 'cat__gender_Female', 'cat__gender_Male',
       'cat__Partner_No', 'cat__Partner_Yes', 'cat__Dependents_No',
       'cat__Dependents_Yes', 'cat__PhoneService_No',
       'cat__PhoneService_Yes', 'cat__MultipleLines_No',
       'cat__MultipleLines_No phone service', 'cat__MultipleLines_Yes',
       'cat__InternetService_DSL', 'cat__InternetService_Fiber optic',
       'cat__InternetService_No', 'cat__OnlineSecurity_No',
       'cat__OnlineSecurity_No internet service',
       'cat__OnlineSecurity_Yes', 'cat__OnlineBackup_No',
       'cat__OnlineBackup_No internet service', 'cat__OnlineBackup_Yes',
       'cat__DeviceProtection_No',
       'cat__DeviceProtection_No internet service',
       'cat__DeviceProtection_Yes', 'cat__TechSupport_No',
       'cat__TechSupport_No internet service', 'cat__TechSupport_Yes',
       'cat__StreamingTV_No', 'cat__StreamingTV_No internet service',
       'cat__